# Introduzione a NumPy e PyTorch


## Perché ci servono NumPy e PyTorch?

Il machine learning, sotto il cofano, è **algebra lineare su tantissimi numeri**: moltiplicare matrici, sommare vettori, calcolare medie e derivate. Python "puro" (liste e cicli `for`) è comodo ma **lento** per questo tipo di lavoro.

| Libreria | In una frase | La useremo per |
|---|---|---|
| **NumPy** | Array di numeri veloci (scritti in C sotto il cofano) | Preparare i dati, costruire un modello "a mano" |
| **PyTorch** | Come NumPy, ma con **GPU** e **calcolo automatico delle derivate** | Costruire e addestrare reti neurali |


##NumPy
NumPy fornisce un oggetto principale: l'**array** (`np.ndarray`), una griglia di numeri tutti dello stesso tipo.

**Perché è veloce?**
- I numeri sono salvati **uno dopo l'altro in memoria**, come in un array C.
- Le operazioni sono **vettorizzate**: si scrive `a + b` e il ciclo viene eseguito in C, non in Python.

In [ ]:
import numpy as np
import time

print("Versione NumPy:", np.__version__)

### Lista Python vs array NumPy

In [ ]:
lista = [1, 2, 3, 4]
arr = np.array([1, 2, 3, 4])

print("lista * 2 =", lista * 2)   # ripete la lista!
print("arr * 2   =", arr * 2)     # moltiplica ogni elemento

###Come creare un array

In [ ]:
print(np.zeros(5))                 # 5 zeri
print(np.ones((2, 3)))             # matrice 2x3 di uni
print(np.arange(0, 10, 2))         # come range(): 0, 2, 4, 6, 8
print(np.linspace(0, 1, 5))        # 5 numeri equidistanti tra 0 e 1

rng = np.random.default_rng(42)    # generatore casuale con "seed" fisso -> risultati riproducibili
print(rng.normal(size=(2, 3)))     # numeri casuali da una gaussiana

### Operazioni e aggregazioni (`axis`)


In [ ]:
v = np.array([1.0, 2.0, 3.0])          # vettore
M = np.array([[1, 2, 3],
              [4, 5, 6]])               # matrice 2 righe x 3 colonne


print("Somma totale:      ", M.sum())
print("Somma per colonna: ", M.sum(axis=0))   # axis=0 -> "schiaccia" le righe
print("Media per riga:    ", M.mean(axis=1))  # axis=1 -> "schiaccia" le colonne

### Broadcasting

NumPy sa combinare array di shape diverse "allargando" quello più piccolo. Esempio: sottrarre a ogni colonna la sua media.

In [ ]:
X = np.array([[1.0, 200.0],
              [2.0, 400.0],
              [3.0, 600.0]])          # shape (3, 2): 3 campioni, 2 feature

media = X.mean(axis=0)               # shape (2,)
std = X.std(axis=0)                  # shape (2,)

X_norm = (X - media) / std           # (3,2) - (2,) -> la media viene "copiata" su ogni riga
print("media:", media)
print(X_norm)

### Prodotto matriciale: `@`

Un modello lineare calcola, per ogni campione, `y = x·w + b`. Con le matrici lo facciamo per **tutti i campioni in una volta**:

In [ ]:
rng = np.random.default_rng(42)    # generatore casuale con "seed" fisso -> risultati riproducibili
X = rng.normal(size=(4, 3))     # 4 campioni, 3 feature
w = np.array([0.5, -1.0, 2.0])  # un peso per feature
b = 0.1

y = X @ w + b                   # (4,3) @ (3,) -> (4,)  una predizione per campione
print("shape:", y.shape)
print(y)

##PyTorch
PyTorch è una libreria per il deep learning. Possiamo pensarla come **NumPy + tre superpoteri**:

1. **GPU**: gli stessi calcoli possono girare su scheda grafica (molto più veloce per reti grandi).
2. **Autograd**: calcola **automaticamente le derivate** (i gradienti) di qualsiasi calcolo. È il motivo principale per cui la usiamo.
3. **Mattoncini pronti** (`torch.nn`, `torch.optim`): layer, funzioni di loss, ottimizzatori.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("Versione PyTorch:", torch.__version__)

### Tensori: gli "array" di PyTorch

Un **tensore** (`torch.Tensor`) è quasi identico a un array NumPy.

In [ ]:
t = torch.tensor([[1.0, 2.0],
                  [3.0, 4.0]])

print(t)
print("shape:", t.shape, "| dtype:", t.dtype)
print("t @ t =\n", t @ t)
print("media per colonna:", t.mean(dim=0))    # in PyTorch si chiama dim invece di axis

### Da NumPy a PyTorch e ritorno

In [ ]:
arr = np.array([1.0, 2.0, 3.0])

t1 = torch.tensor(arr)        # COPIA i dati
t2 = torch.from_numpy(arr)    # CONDIVIDE la memoria con arr
back = t1.numpy()             # da tensore ad array

print(t1.dtype)               # float64: NumPy usa float64 di default
print(t1.float().dtype)       # float32: lo standard nel deep learning (più veloce, meno memoria)

### CPU o GPU (device)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Stiamo usando:", device)

Su Colab la GPU si attiva da *Runtime → Cambia tipo di runtime*. **Per i dati tabellari di oggi la CPU basta e avanza.**

### Autograd: le derivate automatiche

Prendiamo un modello minuscolo `y = w·x + b` e una loss `L = (y - y_vero)²`.
Vogliamo sapere **come cambia la loss se cambiamo w e b**, cioè le derivate ∂L/∂w e ∂L/∂b.


In [ ]:
x = torch.tensor(3.0)
y_vero = torch.tensor(10.0)

w = torch.tensor(2.0, requires_grad=True)   # requires_grad=True -> "traccia questa variabile"
b = torch.tensor(1.0, requires_grad=True)

y_pred = w * x + b                 # forward: 2*3 + 1 = 7
loss = (y_pred - y_vero) ** 2      # (7 - 10)^2 = 9

loss.backward()                    # backward: PyTorch calcola tutte le derivate

print("loss   =", loss.item())
print("dL/dw  =", w.grad.item())
print("dL/db  =", b.grad.item())

Verifichiamo a mano con la regola della catena:

- ∂L/∂w = 2·(y_pred − y_vero)·x = 2·(7 − 10)·3 = **−18**
- ∂L/∂b = 2·(y_pred − y_vero)·1 = 2·(7 − 10) = **−6**
> **Il punto chiave:** per questo esempio la derivata si fa a mano in 10 secondi. Per una rete con milioni di parametri è impossibile: PyTorch la calcola per noi con `loss.backward()`.

## Riepilogo: NumPy vs PyTorch

| | NumPy | PyTorch |
|---|---|---|
| Oggetto base | `np.ndarray` | `torch.Tensor` |
| Creazione | `np.array(...)` | `torch.tensor(...)` |
| Dimensione per aggregare | `axis=` | `dim=` |
| dtype di default | `float64` | `float32` |
| GPU | ❌ | ✅ `.to("cuda")` |
| Derivate automatiche | ❌ (a mano) | ✅ `loss.backward()` |
| Layer, loss, ottimizzatori pronti | ❌ | ✅ `torch.nn`, `torch.optim` |
| Quando usarla | Preparare dati, calcoli numerici | Modelli che si addestrano |